# Maintenance Work Order Data Workflow

This notebook develops a reproducible Python data workflow for a synthetic CMMS-style maintenance dataset. The records are simulated for educational purposes and do not represent real customer, technician, property, or asset data.

The current project phase focuses on data generation, ingestion, cleaning, exploratory analysis, visualization, and interpretation as a foundation for later machine-learning work-order priority prediction.


## 1. Setup

Import the Python libraries used in the workflow and make the repository root available on the Python import path.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import RAW_DATA_FILE, PROCESSED_DATA_FILE
from src.generate_data import save_dataset
from src.cleaning import (
    standardize_categories,
    remove_duplicate_work_orders,
    flag_invalid_values,
    impute_missing_values,
)
from src.eda import (
    PRIORITY_ORDER,
    priority_summary,
    asset_type_summary,
    priority_operational_summary,
    high_risk_work_orders,
)
from src.visualizations import (
    plot_priority_distribution,
    plot_failures_by_priority,
    plot_median_cost_by_asset_type,
    plot_resolution_by_priority,
)

pd.set_option('display.max_columns', None)
print(f'Project root: {project_root}')
print(f'Raw data path: {RAW_DATA_FILE}')
print(f'Processed data path: {PROCESSED_DATA_FILE}')


## 2. Data Ingestion

The raw CSV is generated reproducibly when it is not already present. The dataset is then loaded with Pandas, and the first rows are displayed as required by the project rubric.

The parser uses `keep_default_na=False` with `na_values=['']`. This is important because **`None` is a valid value for `occupancy_impact`**, but Pandas can otherwise interpret the literal word `None` as a missing value. Empty CSV fields are still treated as missing data.


In [ ]:
if not RAW_DATA_FILE.exists():
    generated_df = save_dataset()
    print(f'Generated {len(generated_df):,} synthetic raw rows.')
else:
    print('Using the existing synthetic raw CSV.')

df_raw = pd.read_csv(
    RAW_DATA_FILE,
    parse_dates=['created_date'],
    keep_default_na=False,
    na_values=[''],
)
print(f'Loaded {df_raw.shape[0]:,} rows and {df_raw.shape[1]} columns.')
df_raw.head()


### Initial Structure Check

Before cleaning, inspect dimensions, data types, missing values, duplicate work-order IDs, and category counts. This establishes what problems the cleaning functions need to address.


In [ ]:
print('Shape:', df_raw.shape)
print('\nData types:')
display(df_raw.dtypes.to_frame('dtype'))

print('Missing values:')
display(
    df_raw.isna().sum()
    .loc[lambda s: s > 0]
    .sort_values(ascending=False)
    .to_frame('missing_count')
)

duplicate_ids = df_raw.duplicated(subset='work_order_id').sum()
print(f'Duplicate work-order IDs: {duplicate_ids}')
print(f'Raw asset-type labels: {df_raw["asset_type"].nunique()}')
print(f'Raw priority labels: {df_raw["priority"].nunique()}')
print(f'Rows whose occupancy impact is the valid category "None": {(df_raw["occupancy_impact"] == "None").sum()}')


## 3. Data Cleaning

The raw dataset intentionally contains missing values, inconsistent capitalization and whitespace, duplicate work orders, and a small number of impossible or injected out-of-range values. Four reusable cleaning functions in `src/cleaning.py` are applied below. Every function includes an informative docstring.

### Cleaning decisions and justification

1. **Standardize categories.** Equivalent labels such as `HVAC`, `hvac`, and ` HVAC ` must represent one category. Without normalization, grouping and later ML encoding would incorrectly treat them as different values.
2. **Remove duplicate work orders.** Duplicate work-order IDs would double-count maintenance events and bias frequency, cost, and priority summaries.
3. **Flag invalid values.** Negative asset ages are impossible. Resolution times above 240 hours and repair costs above $25,000 are outside the documented valid range of this synthetic generator and were intentionally injected as quality problems. They are changed to missing values before imputation.
4. **Impute missing values.** Missing asset condition is labeled `Unknown` instead of inventing a Good/Fair/Poor condition. Missing numerical values are filled using the median for the same asset type and maintenance type, with an overall median fallback. This preserves rows while limiting sensitivity to skewed cost and duration distributions.

These thresholds and imputation rules are appropriate for this **synthetic educational dataset**. They must be re-evaluated rather than copied blindly when real client data becomes available.


In [ ]:
raw_quality = pd.Series({
    'rows': len(df_raw),
    'duplicate_work_order_ids': df_raw.duplicated(subset='work_order_id').sum(),
    'missing_values': df_raw.isna().sum().sum(),
    'unique_asset_type_labels': df_raw['asset_type'].nunique(),
    'unique_priority_labels': df_raw['priority'].nunique(),
    'negative_asset_ages': (df_raw['asset_age_years'] < 0).sum(),
    'resolution_hours_over_240': (df_raw['resolution_hours'] > 240).sum(),
    'repair_cost_over_25000': (df_raw['estimated_repair_cost'] > 25000).sum(),
}, name='raw')

df_clean = standardize_categories(df_raw)
df_clean = remove_duplicate_work_orders(df_clean)
df_clean = flag_invalid_values(df_clean)
df_clean = impute_missing_values(df_clean)

clean_quality = pd.Series({
    'rows': len(df_clean),
    'duplicate_work_order_ids': df_clean.duplicated(subset='work_order_id').sum(),
    'missing_values': df_clean.isna().sum().sum(),
    'unique_asset_type_labels': df_clean['asset_type'].nunique(),
    'unique_priority_labels': df_clean['priority'].nunique(),
    'negative_asset_ages': (df_clean['asset_age_years'] < 0).sum(),
    'resolution_hours_over_240': (df_clean['resolution_hours'] > 240).sum(),
    'repair_cost_over_25000': (df_clean['estimated_repair_cost'] > 25000).sum(),
}, name='clean')

cleaning_comparison = pd.concat([raw_quality, clean_quality], axis=1)
cleaning_comparison


### Cleaning result and bias considerations

The comparison above demonstrates the effect of the cleaning functions rather than hiding the changes. Duplicate rows are removed, categorical labels collapse to their canonical groups, impossible synthetic values are handled, and missing values are resolved.

Cleaning can itself introduce bias. For example, deleting every row with a missing field could disproportionately remove certain asset types or priority classes if documentation quality differs by group. This workflow therefore preserves rows where possible. Even median imputation is not neutral: it reduces variability and can pull unusual records toward a typical group value. For that reason, the original raw data is kept unchanged, the cleaning rules are explicit, and the project documents that these rules must be validated on real operational data before production use.


In [ ]:
PROCESSED_DATA_FILE.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(PROCESSED_DATA_FILE, index=False)
print(f'Saved {len(df_clean):,} cleaned rows to {PROCESSED_DATA_FILE}')
df_clean.head()


## 4. Exploratory Data Analysis

EDA is performed on the cleaned data with reusable functions from `src/eda.py`. The goal is to understand distributions and relationships that may later inform feature engineering. Because the data is synthetic, these patterns demonstrate the workflow and the assumptions built into the generator; they are not evidence about real facilities.


In [ ]:
priority_table = priority_summary(df_clean)
priority_table


**EDA interpretation:** The priority table makes the class imbalance explicit. Medium work orders are the largest group and Emergency work orders are intentionally uncommon. This matters for future classification because accuracy alone could hide poor performance on the smaller Emergency class.


In [ ]:
asset_summary = asset_type_summary(df_clean)
asset_summary


**EDA interpretation:** Work-order volume and cost differ across asset categories. Some of this structure is intentional in the synthetic generator—for example, asset categories have different base repair costs and criticality profiles—so the table is useful for testing a real workflow but should not be generalized to actual portfolios.


In [ ]:
priority_ops = priority_operational_summary(df_clean)
priority_ops


**EDA interpretation:** Comparing failure history, PM lateness, costs, safety-related share, and resolution time across priority groups helps identify candidate predictors and potential leakage. Failure history and PM information are available before resolution and can be considered later as features. `resolution_hours` is observed only after completion, so it must not be used to predict priority at intake.


In [ ]:
risk_subset = high_risk_work_orders(
    df_clean,
    minimum_failures=3,
    minimum_overdue_days=30,
)
print(f'Work orders meeting the exploratory repeated-failure + overdue-PM filter: {len(risk_subset):,}')
risk_subset.head(10)


**EDA interpretation:** This transparent filter identifies records that combine repeated recent failures with overdue preventive maintenance. It is not a predictive risk score; it is simply a way to inspect a potentially important operational segment and demonstrate filtering as part of EDA.


In [ ]:
numeric_columns = [
    'asset_age_years',
    'previous_failures_12m',
    'days_since_last_service',
    'pm_overdue_days',
    'estimated_repair_cost',
    'resolution_hours',
]
df_clean[numeric_columns].describe().round(2)


## 5. Visualizations

The following Matplotlib visualizations turn key EDA findings into interpretable charts. Each chart has a descriptive title and labeled axes, followed by a short interpretation.


In [ ]:
plot_priority_distribution(df_clean)
plt.show()


**Visualization 1 interpretation:** Medium is the most common priority and Emergency is the least common. The imbalance is deliberate and more realistic than four equally sized classes. A future classifier should therefore be evaluated with class-sensitive measures such as precision, recall, F1 score, and a confusion matrix rather than accuracy alone.


In [ ]:
plot_failures_by_priority(df_clean)
plt.show()


**Visualization 2 interpretation:** Previous-failure averages vary across priority groups, illustrating why maintenance history may be informative for future priority prediction. In this synthetic dataset, failure history contributes to the generated urgency relationship, so this is an intentionally designed association rather than proof of a real-world causal effect.


In [ ]:
plot_median_cost_by_asset_type(df_clean)
plt.show()


**Visualization 3 interpretation:** Median estimated repair cost differs by asset type. These differences are partly built into the synthetic data-generation assumptions and demonstrate why categorical asset information can be important in maintenance analytics. With real data, the same plot would need to be interpreted in the context of asset mix, labor practices, parts costs, and portfolio size.


In [ ]:
plot_resolution_by_priority(df_clean)
plt.show()


**Visualization 4 interpretation:** Resolution-time distributions overlap across all priority levels, while their typical values can still differ. This is useful operationally but must be treated carefully in machine learning: resolution time occurs after the priority decision and would create target leakage if used as an intake predictor.


## 6. Summary and Interpretation

The workflow started with a deliberately imperfect synthetic CMMS dataset and produced a cleaned, reusable dataset suitable for exploratory analysis and future machine-learning preparation. Cleaning consolidated inconsistent categorical labels, removed duplicate work orders, handled impossible injected values, and imputed missing data while preserving records whenever reasonable.

EDA shows a deliberately imbalanced priority distribution, differences among asset categories, and associations between operational variables such as failure history, preventive-maintenance lateness, safety indicators, cost, and work-order priority. These patterns are useful for demonstrating how maintenance data can be inspected before model design. They also highlight candidate future features such as asset type, age, condition, criticality, prior failures, PM compliance, occupancy impact, and safety status.

Several assumptions are important. The dataset was generated from domain-informed rules and controlled randomness rather than observed customer operations. Therefore, relationships found here partly reflect the assumptions encoded by the generator. They demonstrate the analysis process, not validated real-world causal relationships. The cleaning thresholds are also specific to this synthetic dataset and would need to be re-estimated from actual business rules and observed distributions.

The most important limitation is generalizability: real CMMS data may contain different failure modes, missingness patterns, work-order practices, asset taxonomies, technician behavior, and priority definitions. Once real operational data becomes available, the same workflow should be rerun with a new data audit, revised cleaning rules, bias checks, and validation of every feature. For future priority prediction, post-outcome fields such as `resolution_hours` must be excluded to prevent data leakage.

Overall, the project now provides a professional foundation for later ML/DL work: reproducible ingestion, documented cleaning, reusable analysis functions, interpreted visualizations, explicit assumptions, bias awareness, and a processed dataset that can be used for feature engineering and model evaluation in the next phase.
